# 02 — Filter and epoch imagery EEG
Filtering selects a frequency range; epoching cuts cue-aligned trials from continuous recordings. Neither operation demonstrates classification. This notebook reloads the same subject and uses the shared experiment configuration.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Launch Jupyter from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
config = json.loads((ROOT / "experiments/experiment_001/config.json").read_text())
from src.data.load_eeg import load_eeg
from src.features.spectral import compute_psd
from src.visualization.eeg_plots import plot_raw_segment
print("MNE", mne.__version__, "subject", config["subject"], "runs", config["runs"])
raw = load_eeg(config["subject"], config["runs"], ROOT / "data/raw")

## Inspect task durations
Confirm the selected window stays inside imagery periods. Never silently interpret resting samples as imagery.

In [ ]:
task = np.isin(raw.annotations.description, ["T1", "T2"])
assert raw.annotations.duration[task].min() >= config["tmax"]

## Apply 8–30 Hz filtering
This exploratory band retains mu/alpha and beta activity. A zero-phase FIR filter uses past and future samples, so this is an offline analysis. Filtering continuous data reduces epoch-edge effects; run boundaries are handled separately. We retain the recorded reference and apply no automatic ICA or artifact rejection.

In [ ]:
from src.preprocessing.filters import bandpass_filter
filtered = bandpass_filter(raw, config["l_freq"], config["h_freq"])
plot_raw_segment(filtered, start=10., duration=5.)
plt.show()
compute_psd(filtered, fmin=1., fmax=60.).plot(average=True, show=False)
plt.show()

## Create trials
Each trial contains 1–4 seconds after the imagery cue, with no mean baseline subtraction. Omitting the first second is a starting assumption about cue transients. MNE rejects windows crossing BAD boundaries. Retained metadata includes run identity for later validation.

In [ ]:
from src.preprocessing.epochs import create_imagery_epochs
from src.preprocessing.artifacts import epoch_counts, drop_summary
epochs = create_imagery_epochs(filtered, config["runs"], tmin=config["tmin"],
                               tmax=config["tmax"], baseline=config["baseline"],
                               reject=config["reject"])
print(epochs)
print("Candidate epochs:", len(epochs.drop_log))
display(epoch_counts(epochs))
display(drop_summary(epochs))
display(epochs.metadata.groupby(["run", "condition"]).size().rename("n_epochs"))

## Look at example trials
Individual trials are noisy. Compare both conditions without interpreting one example as a systematic difference. Epoch amplitude is shown in µV.

In [ ]:
from src.visualization.eeg_plots import plot_example_epoch
for condition in epochs.event_id:
    plot_example_epoch(epochs, condition)
plt.show()

## Descriptive condition PSD
Average Welch PSD across trials at C3 and C4 for each condition. Variation across trials and artifacts may explain apparent differences. This is neither a statistical test nor ERD/ERS relative to rest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for condition in epochs.event_id:
    spec = compute_psd(epochs[condition], fmin=8., fmax=30.)
    values, freqs = spec.get_data(return_freqs=True)
    for ax, channel in zip(axes, ["C3", "C4"]):
        idx = spec.ch_names.index(channel)
        density = values[:, idx, :].mean(axis=0)
        ax.plot(freqs, 10 * np.log10(np.maximum(density * 1e12, np.finfo(float).tiny)), label=condition)
        ax.set(title=channel, xlabel="Frequency (Hz)")
        ax.legend()
axes[0].set_ylabel("PSD (dB re 1 µV²/Hz)")
fig.tight_layout()
plt.show()

## Optional local save
Set SAVE=True to save derived epochs. Raw data stays unchanged; both raw and processed data are excluded from Git. Saving is not required to finish the milestone.

In [ ]:
SAVE = False
if SAVE:
    epochs.save(ROOT / "data/processed/subject_001-imagery-epo.fif", overwrite=True)

## Milestone checklist
Confirm retrieval, 64 named EEG channels, sampling frequency, correct T1/T2 mapping, valid task durations, filter parameters, per-class/per-run epoch counts, drop reasons and plots. Record problems and assumptions. Stop here: no classifier has been fitted and neither hypothesis has been tested.